# Baseline — Cadastros suspeitos de clientes

*Dupla:* Emilly e Cleber

## Objetivo
Treinar um primeiro modelo Random Forest para identificar cadastros
sinalizados pelas regras selecionadas da Silver.

## Dados e preparação
A preparação é executada pelo notebook FE_clientes_cadastros_suspeitos:
- Leitura das tabelas Silver de clientes e endereços.
- Criação dos atributos de e-mail, hash, idade e endereços.
- Consolidação de cópias idênticas.
- Criação do rótulo experimental rotulo_revisao_v2.
- Separação temporal entre treino e avaliação.
- Codificação dos domínios usando somente o treino.

## Avaliação
Os lotes a partir de 16/10/2026 ficam reservados para avaliação.

O rótulo indica sinalização pelas regras, não confirmação de fraude.
As métricas medirão concordância com essas regras.

In [0]:
%run ./FE_clientes_cadastros_suspeitos

In [0]:
#============================================================
# CONFERÊNCIA DAS ENTRADAS DO BASELINE
# ============================================================

variaveis_necessarias = [
    "df_treino_codificado",
    "df_avaliacao_codificada",
    "features_finais",
    "alvo_modelo",
]

faltantes = [
    nome
    for nome in variaveis_necessarias
    if nome not in globals()
]

if faltantes:
    raise ValueError(
        f"Variáveis ausentes: {faltantes}. "
        "Execute o notebook de engenharia de features."
    )

features_baseline = list(features_finais)

if not features_baseline:
    raise ValueError("A lista de atributos está vazia.")

if len(features_baseline) != len(set(features_baseline)):
    raise ValueError("Existem atributos repetidos na lista.")

for nome, df in [
    ("TREINO", df_treino_codificado),
    ("AVALIAÇÃO", df_avaliacao_codificada),
]:
    colunas_ausentes = [
        coluna
        for coluna in features_baseline + [alvo_modelo]
        if coluna not in df.columns
    ]

    if colunas_ausentes:
        raise ValueError(
            f"Colunas ausentes em {nome}: {colunas_ausentes}"
        )

    total = df.count()

    if total == 0:
        raise ValueError(f"A base {nome} está vazia.")

    print(f"{nome}: {total} registros")

print("\n===== ATRIBUTOS DO BASELINE =====")
for coluna in features_baseline:
    print(f"- {coluna}")

print("\nQuantidade de atributos:", len(features_baseline))
print("Alvo:", alvo_modelo)
print("\nEntradas disponíveis para o treinamento.")
print("Nenhum modelo foi treinado por esta célula.")

In [0]:
# ============================================================
# TREINAMENTO DO BASELINE RANDOM FOREST
# ============================================================

import numpy as np
from importlib.metadata import version
from sklearn.ensemble import RandomForestClassifier

# 1. Verificar as variáveis necessárias
variaveis_necessarias = [
    "df_treino_codificado",
    "features_finais",
    "alvo_modelo",
]

faltantes = [
    nome
    for nome in variaveis_necessarias
    if nome not in globals()
]

if faltantes:
    raise ValueError(
        f"Variáveis ausentes: {faltantes}. "
        "Execute as células 2 e 3."
    )

versao_sklearn = version("scikit-learn")
features_baseline = list(features_finais)

# 2. Validar a lista de atributos
if not features_baseline:
    raise ValueError("A lista de atributos está vazia.")

if len(features_baseline) != len(set(features_baseline)):
    raise ValueError("Existem atributos repetidos na lista.")

colunas_proibidas = {
    "id_cliente",
    "bronze_ingested_at",
    "bronze_source_file",
    "dominio_email",
    "regra_04_falhou",
    "regra_05_falhou",
    "regra_09_falhou",
    "motivo_hash",
    "motivo_data_silver",
    "motivo_dominio",
    alvo_modelo,
}

indevidas = sorted(
    set(features_baseline).intersection(colunas_proibidas)
)

if indevidas:
    raise ValueError(
        f"Colunas indevidas entre os preditores: {indevidas}"
    )

colunas_necessarias = (
    ["id_cliente", "bronze_ingested_at"]
    + features_baseline
    + [alvo_modelo]
)

colunas_ausentes = [
    coluna
    for coluna in colunas_necessarias
    if coluna not in df_treino_codificado.columns
]

if colunas_ausentes:
    raise ValueError(
        f"Colunas ausentes no treino: {colunas_ausentes}"
    )

# 3. Converter somente a base de treino para pandas
# O limite evita uma coleta excessiva para a memória do driver.
limite_registros = 100_000

treino_pd = (
    df_treino_codificado
    .orderBy("id_cliente", "bronze_ingested_at")
    .select(*features_baseline, alvo_modelo)
    .limit(limite_registros + 1)
    .toPandas()
)

if len(treino_pd) > limite_registros:
    raise ValueError(
        "O treino ultrapassa o limite de coleta para pandas."
    )

if treino_pd.empty:
    raise ValueError("A base de treino está vazia.")

X_treino = treino_pd[features_baseline].astype("float64")
y_treino = treino_pd[alvo_modelo]

# 4. Validar os valores antes do treinamento
if not np.isfinite(X_treino.to_numpy()).all():
    raise ValueError(
        "Existem valores nulos, NaN ou infinitos nos atributos."
    )

if y_treino.isna().any() or not y_treino.isin([0, 1]).all():
    raise ValueError(
        "O alvo deve conter somente 0 e 1, sem valores nulos."
    )

y_treino = y_treino.astype("int64")

if set(y_treino.unique()) != {0, 1}:
    raise ValueError(
        "A base de treino precisa conter as duas classes."
    )

# 5. Definir os parâmetros do primeiro modelo
parametros_rf = {
    "n_estimators": 100,
    "max_depth": 5,
    "min_samples_leaf": 2,
    "max_features": "sqrt",
    "class_weight": "balanced",
    "random_state": 42,
    "n_jobs": 1,
}

# 6. Treinar usando somente os dados de treino
modelo_rf = RandomForestClassifier(**parametros_rf)
modelo_rf.fit(X_treino, y_treino)

# 7. Exibir o resumo do treinamento
print("===== RANDOM FOREST TREINADO =====")
print("Versão do scikit-learn:", versao_sklearn)
print("Registros de treino:", len(X_treino))
print("Quantidade de atributos:", len(features_baseline))
print("Alvo:", alvo_modelo)

print("\n===== CLASSES NO TREINO =====")
for classe, quantidade in y_treino.value_counts().sort_index().items():
    print(f"Classe {classe}: {quantidade} registros")

print("\n===== PARÂMETROS =====")
for parametro, valor in parametros_rf.items():
    print(f"{parametro}: {valor}")

print("\n===== ATRIBUTOS UTILIZADOS =====")
for coluna in features_baseline:
    print(f"- {coluna}")

print("\nOs dados de avaliação não foram usados no treinamento.")
print("O modelo aprende a reproduzir o rótulo experimental.")
print("Esse rótulo não representa confirmação de fraude.")
print("Nenhum dado da Silver foi alterado.")
print("O modelo não foi salvo explicitamente por esta célula.")

In [0]:
# ============================================================
# AVALIAÇÃO DO BASELINE
# Avaliação nos lotes reservados, sem novo treinamento.
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    average_precision_score,
    roc_auc_score,
)

# 1. Validar as dependências
necessarias = [
    "modelo_rf",
    "df_avaliacao_codificada",
    "features_baseline",
    "alvo_modelo",
]

faltantes = [
    nome for nome in necessarias
    if nome not in globals()
]

if faltantes:
    raise ValueError(
        f"Variáveis ausentes: {faltantes}. "
        "Execute as células anteriores."
    )

if list(modelo_rf.feature_names_in_) != list(features_baseline):
    raise ValueError(
        "A lista ou a ordem dos atributos difere do treinamento."
    )

# Limiar definido antes de consultar os resultados.
limiar_baseline = 0.5

# 2. Preparar a avaliação na mesma ordem de atributos do treino
limite_avaliacao = 100_000

avaliacao_pd = (
    df_avaliacao_codificada
    .orderBy("id_cliente", "bronze_ingested_at")
    .select(
        "id_cliente",
        "bronze_ingested_at",
        *features_baseline,
        alvo_modelo,
    )
    .limit(limite_avaliacao + 1)
    .toPandas()
)

if len(avaliacao_pd) > limite_avaliacao:
    raise ValueError("Avaliação acima do limite de coleta para pandas.")

if avaliacao_pd.empty:
    raise ValueError("A base de avaliação está vazia.")

X_avaliacao = avaliacao_pd[features_baseline].astype("float64")
y_avaliacao = avaliacao_pd[alvo_modelo]

if not np.isfinite(X_avaliacao.to_numpy()).all():
    raise ValueError(
        "Existem valores nulos, NaN ou infinitos nos atributos."
    )

if (
    y_avaliacao.isna().any()
    or not y_avaliacao.isin([0, 1]).all()
):
    raise ValueError("O alvo deve conter somente 0 e 1, sem nulos.")

y_avaliacao = y_avaliacao.astype("int64")

# 3. Gerar os escores e as classificações
indice_classe_1 = list(modelo_rf.classes_).index(1)

escores_avaliacao = modelo_rf.predict_proba(
    X_avaliacao
)[:, indice_classe_1]

predicoes_avaliacao = (
    escores_avaliacao >= limiar_baseline
).astype("int64")

# Referência simples: classificar todos como não sinalizados.
predicoes_referencia = np.zeros(len(y_avaliacao), dtype="int64")

# 4. Comparar as métricas da classe 1
def calcular_metricas(nome, predicoes):
    return {
        "modelo": nome,
        "acuracia": accuracy_score(y_avaliacao, predicoes),
        "precisao_classe_1": precision_score(
            y_avaliacao, predicoes, zero_division=0
        ),
        "recall_classe_1": recall_score(
            y_avaliacao, predicoes, zero_division=0
        ),
        "f1_classe_1": f1_score(
            y_avaliacao, predicoes, zero_division=0
        ),
    }

metricas_baseline = pd.DataFrame([
    calcular_metricas("Random Forest", predicoes_avaliacao),
    calcular_metricas("Sempre classe 0", predicoes_referencia),
])

print("===== AVALIAÇÃO DO BASELINE =====")
print("Registros:", len(y_avaliacao))
print("Limiar:", limiar_baseline)

print("\n===== CLASSES REAIS =====")
for classe, quantidade in y_avaliacao.value_counts().sort_index().items():
    print(f"Classe {classe}: {quantidade} registros")

print("\n===== MÉTRICAS — ESCALA DE 0 A 1 =====")
display(metricas_baseline.round(4))

# 5. Exibir a matriz de confusão
matriz = confusion_matrix(
    y_avaliacao,
    predicoes_avaliacao,
    labels=[0, 1],
)

matriz_confusao_baseline = pd.DataFrame(
    matriz,
    index=["Real 0", "Real 1"],
    columns=["Previsto 0", "Previsto 1"],
)

print("\n===== MATRIZ DE CONFUSÃO =====")
display(matriz_confusao_baseline.reset_index(names="classe_real"))

tn, fp, fn, tp = matriz.ravel()

print("Verdadeiros negativos:", tn)
print("Falsos positivos:", fp)
print("Falsos negativos:", fn)
print("Verdadeiros positivos:", tp)

# 6. Métricas de ordenação dos escores
if y_avaliacao.nunique() == 2:
    print("\n===== MÉTRICAS DOS ESCORES =====")
    print(
        "Average Precision (AP):",
        round(
            average_precision_score(y_avaliacao, escores_avaliacao),
            4,
        ),
    )
    print(
        "ROC AUC:",
        round(
            roc_auc_score(y_avaliacao, escores_avaliacao),
            4,
        ),
    )
else:
    print("\nAvaliação com uma única classe: AP e ROC AUC não exibidas.")

# 7. Manter os resultados para análise, sem gravar dados
resultados_avaliacao = avaliacao_pd[
    ["id_cliente", "bronze_ingested_at", alvo_modelo]
].copy()

resultados_avaliacao["escore_classe_1"] = escores_avaliacao
resultados_avaliacao["predicao"] = predicoes_avaliacao

print("\nAs métricas medem concordância com o rótulo experimental.")
print("O escore não representa probabilidade comprovada de fraude.")
print("Nenhum parâmetro foi ajustado nesta avaliação.")
print("Nenhum modelo foi retreinado e nenhum dado foi gravado.")

## Conclusão do baseline

O Random Forest foi treinado com 1.063 registros e 17 atributos.
A avaliação temporal utilizou 72 registros de lotes posteriores,
sem compartilhamento de clientes ou arquivos com o treino.

Com limiar de 0,5, o modelo classificou corretamente os 67 registros
da classe 0 e os 5 registros da classe 1, obtendo acurácia,
precisão, recall e F1 de 100% nesta avaliação.

A referência que sempre prevê classe 0 alcançou acurácia de 93,06%,
mas não identificou nenhum registro positivo.

### Limitações
- O rótulo é experimental, derivado de regras da Silver.
- As métricas representam concordância com essas regras,
  não confirmação de fraude.
- A avaliação contém apenas 5 positivos.
- Todos os registros de avaliação possuem endereço disponível.
- O resultado não garante o mesmo desempenho em novos lotes.

### Próximo passo
Preservar esta configuração como baseline e avaliar novos lotes
ou rótulos revisados, sem ajustar parâmetros com base neste teste.